# 🚁 AIC-4 — Exploratory Data Analysis
## Energy-Efficient Drone-Based Aerial Multi-Object Tracking

**Author:** Mohamed Gharieb | Military Technical College AIC-4  
**Purpose:** Understand the dataset characteristics, object distributions, and visual challenges before training.

---

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.notebook import tqdm
import yaml
import json
import warnings
warnings.filterwarnings('ignore')

# Apply dark theme
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
ACCENT = '#00c878'   # project green
PALETTE = ['#00c878', '#00aaff', '#ff6b35', '#ffd700', '#c678ff', '#ff4488']

with open(ROOT / 'configs/config.yaml') as f:
    CFG = yaml.safe_load(f)

print('✅ Environment ready')
print(f"   Classes: {list(CFG['detection']['classes'].values())}")
print(f"   Environments: {CFG['airsim']['environments']}")

## 1. Dataset Overview

In [ ]:
def load_yolo_labels(data_root: Path) -> pd.DataFrame:
    """Load all YOLO-format labels into a DataFrame."""
    rows = []
    ann_dir = data_root / 'annotations'
    if not ann_dir.exists():
        print(f'⚠️  No annotations found at {ann_dir}. Generating synthetic data...')
        return _synthetic_labels()

    for txt in ann_dir.rglob('*.txt'):
        env = txt.parent.name
        for line in txt.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                rows.append({'env': env, 'class_id': cls_id, 'cx': cx,
                             'cy': cy, 'bw': bw, 'bh': bh,
                             'area': bw * bh, 'file': txt.stem})
    return pd.DataFrame(rows) if rows else _synthetic_labels()


def _synthetic_labels() -> pd.DataFrame:
    """Generate synthetic label statistics mirroring expected real data."""
    rng = np.random.default_rng(42)
    envs = CFG['airsim']['environments']
    rows = []
    for env in envs:
        n = rng.integers(1500, 3500)
        for _ in range(n):
            cls = rng.choice([0, 0, 0, 1, 2], p=[0.60, 0.25, 0.15])
            scale = rng.uniform(0.02, 0.12)
            rows.append({'env': env, 'class_id': cls,
                         'cx': rng.uniform(0.05, 0.95),
                         'cy': rng.uniform(0.05, 0.95),
                         'bw': scale, 'bh': scale * rng.uniform(0.5, 2.0),
                         'area': scale ** 2})
    return pd.DataFrame(rows)


DATA_ROOT = ROOT / 'data'
df = load_yolo_labels(DATA_ROOT)
CLASS_NAMES = CFG['detection']['classes']
df['class_name'] = df['class_id'].map(lambda x: CLASS_NAMES.get(int(x), str(x)))

print(f'Total annotations : {len(df):,}')
print(f'Environments      : {df["env"].nunique()}')
print('\nClass distribution:')
print(df['class_name'].value_counts().to_string())

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#111111')

# Global class distribution
counts = df['class_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=PALETTE[:len(counts)], edgecolor='none')
axes[0].set_title('Overall Class Distribution', color='white', fontsize=13)
axes[0].set_ylabel('Count', color='white')
for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_visible(False)

# Per-environment breakdown
pivot = df.groupby(['env', 'class_name']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=axes[1], color=PALETTE[:3], legend=True)
axes[1].set_title('Class Distribution per Environment', color='white', fontsize=13)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].legend(facecolor='#222222', labelcolor='white')

# Pie chart
axes[2].pie(counts.values, labels=counts.index, colors=PALETTE[:len(counts)],
            autopct='%1.1f%%', textprops={'color': 'white'})
axes[2].set_title('Class Proportion', color='white', fontsize=13)

plt.tight_layout()
out = ROOT / 'experiments/results/eda_class_dist.png'
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150, facecolor='#111111')
plt.show()
print(f'Saved → {out}')

## 3. Object Size Analysis (Small-Object Challenge)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#111111')
for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_visible(False)

# Bbox width distribution
for i, (name, grp) in enumerate(df.groupby('class_name')):
    axes[0].hist(grp['bw'] * 640, bins=40, alpha=0.75,
                 label=name, color=PALETTE[i])
axes[0].set_title('Bounding Box Width (pixels @ 640px)', color='white', fontsize=12)
axes[0].set_xlabel('Width (px)', color='white')
axes[0].axvline(20, color='red', linestyle='--', linewidth=1.5, label='SAHI threshold (20px)')
axes[0].legend(facecolor='#222222', labelcolor='white', fontsize=9)

# Bbox area distribution (log scale)
axes[1].hist(df['area'] * (640**2), bins=60, color=ACCENT, edgecolor='none')
axes[1].set_title('Object Area Distribution (px²)', color='white', fontsize=12)
axes[1].set_xlabel('Area (px²)', color='white')
axes[1].axvline(400, color='red', linestyle='--', linewidth=1.5, label='<20×20 threshold')
axes[1].set_yscale('log')
axes[1].legend(facecolor='#222222', labelcolor='white', fontsize=9)

# 2D scatter: width vs height
sample = df.sample(min(2000, len(df)), random_state=42)
for i, (name, grp) in enumerate(sample.groupby('class_name')):
    axes[2].scatter(grp['bw']*640, grp['bh']*640, alpha=0.3, s=8,
                    label=name, color=PALETTE[i])
axes[2].set_title('Width vs Height (px)', color='white', fontsize=12)
axes[2].set_xlabel('Width (px)', color='white')
axes[2].set_ylabel('Height (px)', color='white')
axes[2].legend(facecolor='#222222', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'experiments/results/eda_bbox_size.png', dpi=150, facecolor='#111111')
plt.show()

# Stats summary
small_objs = (df['bw'] * 640 < 20) | (df['bh'] * 640 < 20)
print(f'Objects < 20px wide/tall : {small_objs.sum():,} ({small_objs.mean()*100:.1f}%)')
print('→ SAHI tiling is critical for these objects')

## 4. Spatial Heatmap — Where Do Objects Appear?

In [ ]:
fig, axes = plt.subplots(1, len(CLASS_NAMES) + 1, figsize=(20, 4))
fig.patch.set_facecolor('#111111')

all_hm = np.zeros((32, 32))
for ax in axes:
    ax.set_facecolor('#111111')
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])

for idx, (cid, cname) in enumerate(CLASS_NAMES.items()):
    sub = df[df['class_id'] == int(cid)]
    hm = np.zeros((32, 32))
    xs = np.clip((sub['cx'] * 32).astype(int), 0, 31)
    ys = np.clip((sub['cy'] * 32).astype(int), 0, 31)
    for x, y in zip(xs, ys):
        hm[y, x] += 1
        all_hm[y, x] += 1
    axes[idx].imshow(hm, cmap='plasma', interpolation='bilinear')
    axes[idx].set_title(cname.capitalize(), color='white', fontsize=12)

axes[-1].imshow(all_hm, cmap='inferno', interpolation='bilinear')
axes[-1].set_title('All Objects', color='white', fontsize=12)

plt.suptitle('Spatial Distribution Heatmaps (top-down view)', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'experiments/results/eda_spatial_heatmap.png', dpi=150,
            facecolor='#111111', bbox_inches='tight')
plt.show()
print('✅ Spatial heatmap saved')

## 5. Environment Difficulty Ranking

In [ ]:
# Compute per-environment statistics
env_stats = df.groupby('env').agg(
    total_objects=('class_id', 'count'),
    mean_area=('area', 'mean'),
    small_obj_pct=('bw', lambda x: ((x * 640 < 20).mean() * 100)),
    class_diversity=('class_id', 'nunique')
).reset_index()

# Difficulty score: more objects + smaller + diverse = harder
env_stats['difficulty'] = (
    env_stats['total_objects'] / env_stats['total_objects'].max() * 0.4 +
    env_stats['small_obj_pct'] / env_stats['small_obj_pct'].max() * 0.4 +
    env_stats['class_diversity'] / env_stats['class_diversity'].max() * 0.2
)
env_stats = env_stats.sort_values('difficulty', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#111111')
ax.set_facecolor('#1a1a1a')
bars = ax.barh(env_stats['env'], env_stats['difficulty'],
               color=[PALETTE[i] for i in range(len(env_stats))], edgecolor='none')
ax.set_xlabel('Difficulty Score', color='white')
ax.set_title('Environment Difficulty Ranking', color='white', fontsize=14)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_visible(False)
for bar, val in zip(bars, env_stats['difficulty']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', color='white', fontsize=10)
plt.tight_layout()
plt.savefig(ROOT / 'experiments/results/eda_env_difficulty.png', dpi=150, facecolor='#111111')
plt.show()

print('\nEnvironment Statistics:')
print(env_stats.to_string(index=False))

## 6. Augmentation Preview

In [ ]:
import cv2

def create_synthetic_scene(seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    img = np.full((640, 640, 3), 60, dtype=np.uint8)
    noise = rng.integers(-20, 20, img.shape, dtype=np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    n = rng.integers(5, 15)
    colors = [(0, 200, 80), (0, 100, 220), (220, 80, 0)]
    for _ in range(n):
        x1 = int(rng.integers(10, 580))
        y1 = int(rng.integers(10, 580))
        w = int(rng.integers(15, 60))
        h = int(rng.integers(10, 40))
        cls = int(rng.integers(0, 3))
        cv2.rectangle(img, (x1, y1), (x1+w, y1+h), colors[cls], -1)
    return img

def apply_motion_blur(img: np.ndarray, ksize: int = 7) -> np.ndarray:
    kernel = np.zeros((ksize, ksize))
    kernel[int((ksize - 1)/2), :] = 1.0 / ksize
    return cv2.filter2D(img, -1, kernel)

def apply_perspective(img: np.ndarray, strength: float = 0.1) -> np.ndarray:
    h, w = img.shape[:2]
    d = int(min(h, w) * strength)
    src = np.float32([[0,0],[w,0],[w,h],[0,h]])
    dst = np.float32([[d,d],[w-d,d],[w-d,h-d],[d,h-d]])
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (w, h))

base = create_synthetic_scene(42)
augmentations = [
    ('Original',       base),
    ('Motion Blur',    apply_motion_blur(base, 9)),
    ('Brightness +50', np.clip(base.astype(np.int16) + 50, 0, 255).astype(np.uint8)),
    ('Perspective',    apply_perspective(base, 0.12)),
    ('Flip + Noise',   cv2.flip(base, 1)),
]

fig, axes = plt.subplots(1, len(augmentations), figsize=(20, 4))
fig.patch.set_facecolor('#111111')
for ax, (title, img) in zip(axes, augmentations):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, color='white', fontsize=11)
    ax.axis('off')

plt.suptitle('Data Augmentation Preview', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'experiments/results/eda_augmentation.png', dpi=150,
            facecolor='#111111', bbox_inches='tight')
plt.show()
print('✅ EDA complete — all plots saved to experiments/results/')